# eph_03 — Partial correlations: is kinematics encoding RT-independent?

Are the spike count ~ kinematics correlations found in eph_02 driven by shared
RT variance, or do they reflect independent neural tuning?

**Two complementary analyses:**
1. `spike_count ~ kin | RT` — which kinematic predictors survive after RT is controlled for?
2. `spike_count ~ RT | kin` — does RT encoding persist when each kinematic variable is removed?

## 1. Setup

In [ ]:
%matplotlib inline

import contextlib, io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, OKABE_ITO, style_ax, save_fig
apply_style()

In [ ]:
from pathlib import Path

if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    DATA_ROOT = Path("/root/capsule/data")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    DATA_ROOT = FOR_LOCAL
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "eph_03_partial_corr"
SAVE_FIG = False

print(f"ENV       : {ENV}")
print(f"FOR_LOCAL : {FOR_LOCAL}")

## 2. Data loading

In [ ]:
from data_loading import (
    load_session_quality_filter,
    filter_ephys_units,
    load_units_with_spike_times,
)
import pickle

if ENV == "codeocean":
    base_dirs = [SCRATCH / "session_analysis_mlk"]
    filtered_session_paths = load_session_quality_filter(base_dirs)

    with open(SCRATCH / "combined_unit_tbl.pkl", "rb") as f:
        combined_ephys_data = pickle.load(f)
    filtered_ephys = filter_ephys_units(combined_ephys_data, filtered_session_paths)

    ROOT_SCRATCH = str(DATA_ROOT / "LC-NE_scratch_data_1")
    units_with_spikes = load_units_with_spike_times(filtered_ephys, ROOT_SCRATCH)
else:
    filtered_ephys    = pd.read_pickle(FOR_LOCAL / "filtered_ephys.pkl")
    units_with_spikes = None
    base_dirs         = [FOR_LOCAL]
    print(f"Local dev: filtered_ephys {filtered_ephys.shape}")

## 3. Build trial × unit table

In [ ]:
from ephys_utils import AnalysisConfig, build_all_counts_df

cfg = AnalysisConfig(
    align_key="goCue",
    count_window_s=(0.0, 0.2),
    baseline_window_s=(-1.0, 0.0),
    min_trials_per_group=20,
)

if ENV == "codeocean":
    all_counts_df = build_all_counts_df(units_with_spikes, cfg, base_dirs)
else:
    all_counts_df = pd.read_parquet(FOR_LOCAL / "all_counts_df.parquet")

all_counts_df["first_move_excursion_angle_deg_abs"] = (
    all_counts_df["first_move_excursion_angle_deg"].abs()
)

print("all_counts_df shape:", all_counts_df.shape)

## 4. Imports and predictor list

In [ ]:
from encoding_methods import AnalysisSpec, AnalysisResult, fit_encoding
from encoding_plots import (tstat_hist, t_scatter, registry_heatmap,
                             registry_upset, registry_compare_plot)
from per_unit_stats_registry import PerUnitStatsRegistry
from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import get_session_prefix

KIN_PREDICTORS = {
    "endpoint_x":  "first_move_endpoint_x",
    "endpoint_y":  "first_move_endpoint_y",
    "angle":       "first_move_excursion_angle_deg_abs",
    "peak_vel":    "first_move_peak_velocity",
    "mean_vel":    "first_move_out_mean_velocity",
    "out_peak_v":  "first_move_out_peak_velocity",
    "duration":    "first_move_out_duration",
    "distance":    "first_move_out_total_distance",
}

KIN_QUERY = "reaction_time_firstmove > 0.05 and reaction_time_firstmove < 1.0"
RT_COL    = "reaction_time_firstmove"

## 5. Fit bivariate baselines

`fit_encoding` is cheap — re-fit rather than loading saved results.

In [ ]:
# Re-fit bivariate Spearman for each kinematic predictor.
# fit_encoding is cheap and local — no saved results to load.
biv_specs = [
    AnalysisSpec(
        name=f"biv_{key}",
        predictor_col="spike_count",
        response_col=col,
        method="spearman",
        trial_query=KIN_QUERY,
    )
    for key, col in KIN_PREDICTORS.items()
]

# Also re-fit the primary RT analysis (eph_01 question) for comparison
rt_spec = AnalysisSpec(
    name="biv_rt",
    predictor_col="reaction_time_firstmove",
    response_col="spike_count",
    method="ols",
    trial_query=KIN_QUERY,
    log_x=True, zscore_x=True,
)

with contextlib.redirect_stdout(io.StringIO()):
    biv_results = {spec.name: fit_encoding(all_counts_df, spec) for spec in biv_specs}
    rt_result   = fit_encoding(all_counts_df, rt_spec)

print("Bivariate fits complete:")
for name, res in biv_results.items():
    ns = res.n_sig()
    print(f"  {name:25s}  sig +{ns['pos']:3d} / -{ns['neg']:3d}")
print(f"  {'biv_rt':25s}  sig +{rt_result.n_sig()['pos']:3d} / -{rt_result.n_sig()['neg']:3d}")

## 6. Partial correlations: kin | RT

`AnalysisSpec(method="partial", control_col="reaction_time_firstmove")` computes
Spearman partial correlation using the closed-form rank correlation approach
(df = n − 3).

In [ ]:
# Partial correlation: spike_count ~ kin | RT
# Ask: which kinematic predictors survive after RT is controlled for?
partial_kin_specs = [
    AnalysisSpec(
        name=f"partial_{key}_pRT",
        predictor_col="spike_count",
        response_col=col,
        method="partial",
        control_col=RT_COL,
        trial_query=KIN_QUERY,
        notes=f"Spearman partial: spike_count vs {col} controlling for RT",
    )
    for key, col in KIN_PREDICTORS.items()
]

with contextlib.redirect_stdout(io.StringIO()):
    partial_kin_results = {
        spec.name: fit_encoding(all_counts_df, spec)
        for spec in partial_kin_specs
    }

print("Partial (kin | RT) fits complete:")
for name, res in partial_kin_results.items():
    ns = res.n_sig()
    print(f"  {name:35s}  sig +{ns['pos']:3d} / -{ns['neg']:3d}")

## 7. Partial correlations: RT | kin

For each kinematic variable, remove its shared variance with RT and ask
whether RT encoding persists.

In [ ]:
# Partial correlation: spike_count ~ RT | kin
# Ask: does RT encoding survive after each kinematic variable is controlled for?
# Tests whether RT encoding is independent of movement parameters.
partial_rt_specs = [
    AnalysisSpec(
        name=f"partial_rt_p{key}",
        predictor_col="reaction_time_firstmove",
        response_col="spike_count",
        method="partial",
        control_col=col,
        trial_query=KIN_QUERY,
        log_x=False,  # rank-based — log transform not applicable
        notes=f"Spearman partial: spike_count vs RT controlling for {col}",
    )
    for key, col in KIN_PREDICTORS.items()
]

with contextlib.redirect_stdout(io.StringIO()):
    partial_rt_results = {
        spec.name: fit_encoding(all_counts_df, spec)
        for spec in partial_rt_specs
    }

print("Partial (RT | kin) fits complete:")
for name, res in partial_rt_results.items():
    ns = res.n_sig()
    print(f"  {name:35s}  sig +{ns['pos']:3d} / -{ns['neg']:3d}")

## 8. Register results

In [ ]:
reg = PerUnitStatsRegistry(get_session_prefix=get_session_prefix, alpha=0.05)

# Bivariate baselines
for spec in biv_specs:
    reg.register(biv_results[spec.name])
reg.register(rt_result)

# Partial (kin | RT)
for spec in partial_kin_specs:
    reg.register(partial_kin_results[spec.name])

# Partial (RT | kin)
for spec in partial_rt_specs:
    reg.register(partial_rt_results[spec.name])

print(reg)

## 9. Bivariate vs partial T-stat scatter

Each panel: bivariate T (x) vs partial T (y) controlling for RT.
Points on the diagonal = effect unchanged by RT control.
Points that collapse toward zero = effect was RT-mediated.

In [ ]:
# Bivariate T vs partial T for each kinematic predictor.
# Points that stay far from the diagonal survive RT control.
# Points that collapse toward 0 on the y-axis are explained by RT.

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, (key, col) in zip(axes.flat, KIN_PREDICTORS.items()):
    biv_name     = f"biv_{key}"
    partial_name = f"partial_{key}_pRT"
    t_biv = biv_results[biv_name].stats.set_index(["session_prefix", "unit"])["T"]
    t_par = partial_kin_results[partial_name].stats.set_index(["session_prefix", "unit"])["T"]
    merged = t_biv.rename("biv").to_frame().join(t_par.rename("par"), how="inner").dropna()
    t_scatter(
        merged["biv"], merged["par"],
        xlabel="Bivariate T", ylabel="Partial T (| RT)",
        title=key, diagonal="identity", ax=ax,
    )

fig.suptitle("Bivariate vs partial T-stats: spike_count ~ kin | RT")
plt.tight_layout()
save_fig(fig, "biv_vs_partial_kin", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 10. RT encoding robustness

How many RT-significant units survive after each kinematic variable is
controlled for? A predictor that strongly reduces RT sig suggests shared
neural tuning between RT and that kinematic feature.

In [ ]:
# RT encoding robustness: how many units remain sig after each kinematic control?
# x-axis: bivariate RT sig count
# y-axis: partial RT sig count (controlling for each kin)
biv_rt_n   = rt_result.n_sig()
biv_rt_pos = biv_rt_n["pos"]
biv_rt_neg = biv_rt_n["neg"]

rows = []
for key in KIN_PREDICTORS:
    partial_name = f"partial_rt_p{key}"
    ns = partial_rt_results[partial_name].n_sig()
    rows.append({
        "predictor": key,
        "biv_pos": biv_rt_pos,
        "biv_neg": biv_rt_neg,
        "partial_pos": ns["pos"],
        "partial_neg": ns["neg"],
        "pct_pos_retained": 100 * ns["pos"] / biv_rt_pos if biv_rt_pos else 0,
        "pct_neg_retained": 100 * ns["neg"] / biv_rt_neg if biv_rt_neg else 0,
    })

robustness_df = pd.DataFrame(rows).sort_values("pct_pos_retained", ascending=False)
print("RT encoding retained after controlling for each kinematic variable:")
print(robustness_df[["predictor","partial_pos","partial_neg",
                      "pct_pos_retained","pct_neg_retained"]].to_string(index=False))

x = np.arange(len(robustness_df))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label, color in [
    (axes[0], "pct_pos_retained", "Pos-sig RT units retained (%)", PALETTE["pos"]),
    (axes[1], "pct_neg_retained", "Neg-sig RT units retained (%)", PALETTE["neg"]),
]:
    ax.bar(x, robustness_df[col], color=color, edgecolor="white")
    ax.axhline(100, color=PALETTE["neutral"], ls="--", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(robustness_df["predictor"].tolist(), rotation=40, ha="right")
    ax.set_ylabel(label)
    ax.set_ylim(0, 110)
    style_ax(ax)

fig.suptitle("RT encoding robustness to kinematic controls")
plt.tight_layout()
save_fig(fig, "rt_robustness_to_kin_controls", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 11. Pairwise T-stat correlation heatmaps

Compare structure of bivariate vs partial T-stat matrices.

In [ ]:
# Pairwise T-stat correlation: bivariate vs partial entries
# Shows how much partial correlations shrink relative to bivariate
biv_names     = [f"biv_{k}" for k in KIN_PREDICTORS]
partial_names = [f"partial_{k}_pRT" for k in KIN_PREDICTORS]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
registry_heatmap(reg, entries=biv_names,
                 title="Bivariate T correlations", ax=axes[0])
registry_heatmap(reg, entries=partial_names,
                 title="Partial T correlations (| RT)", ax=axes[1])
plt.tight_layout()
save_fig(fig, "heatmap_biv_vs_partial", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()